# LatentMAS-interp — GPU Test Suite

**Target runtime: ~50 min total. All cells use Qwen3-4B.**

Hard gates (raises SystemExit if failed — do not proceed):
- Gate 1 (~5 min): W_a non-trivial on Qwen3-4B
- Gate 2 (~15 min): Bucket-1 rate ≥ 5% on 50 GSM8K examples

Risk checks (warn but don't stop):
- All 3 tasks run without crash (MBPP+ subprocess)
- Resume skips completed examples
- All 17 conditions run without crash
- Storage projection fits volume

In [ ]:
# 1. Clone / update repo
import os, json, time, shutil
from pathlib import Path
import torch
from collections import Counter

REPO = "https://github.com/spraldev/LatentMAS-interp"
REPO_DIR = "/workspace/LatentMAS-interp"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO} {REPO_DIR}

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
# 2. Install deps
!pip install -q torch transformers accelerate datasets tqdm numpy

In [ ]:
# 3. GPU check
assert torch.cuda.is_available(), "No GPU found — wrong runtime?"
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {vram:.1f} GB")
assert vram >= 20, f"Need ≥20GB VRAM for Qwen3-4B, got {vram:.1f}GB"

In [ ]:
# 4. Logic unit tests — no model, pure CPU math/IO (~30s)
!python test_pipeline.py -v

In [ ]:
# 5. GATE 1: W_a non-trivial on Qwen3-4B (~5 min)
# 1 example, no --smoke so model stays 4B.
# If W_a = identity, Exp A/L/M/Q are dead — stop.
WA_DIR = "/workspace/runs/wa_gate"
os.makedirs(WA_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {WA_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas \
    --tasks gsm8k \
    --max_samples 1 \
    --latent_space_realign

wa_path = Path(WA_DIR) / "wa_matrix.pt"
assert wa_path.exists(), "wa_matrix.pt missing — run failed before writing W_a"

W = torch.load(wa_path, weights_only=False)["W_a"].float()
S = torch.linalg.svdvals(W)
cond  = (S.max() / S.min()).item()
frob  = (W - torch.eye(W.shape[0])).norm().item()
frac1 = ((S - 1).abs() < 0.05).float().mean().item()

print(f"W_a shape:        {tuple(W.shape)}")
print(f"cond number:      {cond:.4f}   (want > 1.5)")
print(f"Frob dist from I: {frob:.4f}   (want > 1.0)")
print(f"frac singvals≈1:  {frac1:.3f}   (want < 0.5)")
print()

if cond > 1.5 and frob > 1.0:
    print("✅ GATE 1 PASSED")
else:
    print("🛑 GATE 1 FAILED — W_a still identity on Qwen3-4B")
    print("   Check --latent_space_realign wiring in final_run.py")
    print("   Paper pivot: drop alignment framing, reframe as negative result")
    raise SystemExit("Gate 1 failed")

In [ ]:
# 6. Core smoke: latent_mas + single_agent on all 3 tasks, 5 ex each (~15 min)
# Tests: forward pass, scoring, file I/O, MBPP+ subprocess exec
SMOKE_DIR = "/workspace/runs/smoke"
os.makedirs(SMOKE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {SMOKE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas single_agent_latent_greedy \
    --tasks gsm8k arc_challenge mbppplus \
    --test \
    --latent_space_realign

print("\n--- Smoke results ---")
PASS = True
for task in ["gsm8k", "arc_challenge", "mbppplus"]:
    for cond in ["latent_mas", "single_agent_latent_greedy"]:
        metas = list(Path(SMOKE_DIR).glob(f"{cond}/{task}/example_*/metadata.json"))
        if not metas:
            print(f"  ❌ {cond}/{task}: NO OUTPUT")
            PASS = False
            continue
        correct = sum(json.loads(m.read_text()).get("correct", False) for m in metas)
        print(f"  ✅ {cond}/{task}: {len(metas)} examples, {correct}/{len(metas)} correct")

assert PASS, "One or more conditions/tasks produced no output — check run.log"

In [ ]:
# 7. GATE 2: Bucket-1 rate ≥ 5% on 50 GSM8K examples (~15 min)
# If B1 < 5% at 50 examples, Exp B/P/Q have no data — stop.
BUCKET_DIR = "/workspace/runs/bucket_gate"
os.makedirs(BUCKET_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {BUCKET_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas single_agent_latent_greedy \
    --tasks gsm8k \
    --max_samples 50 \
    --latent_space_realign

bucket_file = Path(BUCKET_DIR) / "buckets" / "gsm8k.json"
assert bucket_file.exists(), "buckets/gsm8k.json missing — did both conditions run?"

rows = json.loads(bucket_file.read_text())
c = Counter(r["bucket"] for r in rows)
n = len(rows)
b1_rate = c[1] / n

lmas_metas = list(Path(BUCKET_DIR).glob("latent_mas/gsm8k/example_*/metadata.json"))
sa_metas   = list(Path(BUCKET_DIR).glob("single_agent_latent_greedy/gsm8k/example_*/metadata.json"))
lmas_acc = sum(json.loads(m.read_text()).get("correct", False) for m in lmas_metas) / max(len(lmas_metas), 1)
sa_acc   = sum(json.loads(m.read_text()).get("correct", False) for m in sa_metas)   / max(len(sa_metas), 1)

print(f"Buckets (GSM8K, 50ex): B1={c[1]} B2={c[2]} B3={c[3]} B4={c[4]}  n={n}")
print(f"Bucket-1 rate:  {b1_rate*100:.1f}%  (need ≥ 5%)")
print(f"latent_mas:     {lmas_acc*100:.1f}%")
print(f"single_agent:   {sa_acc*100:.1f}%")
print(f"Gap (LMAS-SA):  {(lmas_acc - sa_acc)*100:+.1f}pp")
print()

if b1_rate >= 0.05:
    print("✅ GATE 2 PASSED — enough Bucket-1 examples for Exp B/P/Q")
elif b1_rate > 0:
    print(f"⚠️  GATE 2 MARGINAL — B1 rate {b1_rate*100:.1f}% (below 5% threshold)")
    print("   Exp B/P/Q may have too few examples. Consider running 200 samples before deciding.")
    raise SystemExit("Gate 2 marginal — check before full run")
else:
    print("🛑 GATE 2 FAILED — B1=0 at 50 examples")
    print("   Exp B/P/Q have no data. LatentMAS never beats single-agent on GSM8K.")
    print("   Try arc_challenge or a harder task before abandoning.")
    raise SystemExit("Gate 2 failed")

In [ ]:
# 8. Resume test — second pass must skip completed examples (<30s)
t0 = time.time()
!python final_run.py \
    --output_dir {SMOKE_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions latent_mas single_agent_latent_greedy \
    --tasks gsm8k arc_challenge mbppplus \
    --test \
    --latent_space_realign
elapsed = time.time() - t0

print(f"Second pass: {elapsed:.1f}s")
if elapsed < 60:
    print("✅ Resume OK — skipped all cached examples")
else:
    print("⚠️  Resume SLOW — may be re-running completed examples, check run.log")

In [ ]:
# 9. All 17 conditions smoke — 5 ex, GSM8K only (~10 min)
# Every condition must produce output without crashing
ALL_CONDS_DIR = "/workspace/runs/smoke_allconds"
os.makedirs(ALL_CONDS_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {ALL_CONDS_DIR} \
    --model_name Qwen/Qwen3-4B \
    --conditions all \
    --tasks gsm8k \
    --test \
    --latent_space_realign

print("\n--- All-conditions results ---")
cond_dirs = [d for d in Path(ALL_CONDS_DIR).iterdir()
             if d.is_dir() and d.name not in ("buckets", "logs")]
failed = []
for cond_dir in sorted(cond_dirs):
    metas = list(cond_dir.glob("gsm8k/example_*/metadata.json"))
    if not metas:
        print(f"  ❌ {cond_dir.name}: NO OUTPUT")
        failed.append(cond_dir.name)
    else:
        correct = sum(json.loads(m.read_text()).get("correct", False) for m in metas)
        print(f"  ✅ {cond_dir.name}: {len(metas)} ex, {correct}/{len(metas)} correct")

print()
if failed:
    print(f"⚠️  {len(failed)} conditions produced no output: {failed}")
    print("   Check run.log for tracebacks")
else:
    print("✅ All conditions produced output")

In [ ]:
# 10. Storage projection
def dir_size_gb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e9

smoke_gb = dir_size_gb(ALL_CONDS_DIR)  # 5 ex, all conds, 1 task
projected_gb = smoke_gb * (200 / 5) * 3  # scale to 200 ex, 3 tasks

free_gb  = shutil.disk_usage("/workspace").free  / 1e9
total_gb = shutil.disk_usage("/workspace").total / 1e9

print(f"All-conds smoke disk:  {smoke_gb:.2f} GB  (5 ex, all conds, 1 task)")
print(f"Projected full run:    {projected_gb:.1f} GB  (200 ex, all conds, 3 tasks)")
print(f"Volume free / total:   {free_gb:.0f} / {total_gb:.0f} GB")
print()
if projected_gb > free_gb * 0.8:
    print("⚠️  WARNING: projected usage >80% of free space")
    print("   Add --no_layer_hidden or --no_kv_latent to reduce by ~50%")
else:
    print("✅ Storage OK")

In [ ]:
# 11. Final summary
print("========================================")
print("ALL GATES PASSED — READY FOR FULL RUN")
print("========================================")
print()
print("Check any ⚠️ warnings above before starting.")
print()
FULL_DIR = "/workspace/runs/full"
print("Step 1 — GSM8K + ARC (~7 hr):")
print(f"  python final_run.py --output_dir {FULL_DIR} --model_name Qwen/Qwen3-4B --tasks gsm8k arc_challenge --latent_space_realign")
print()
print("Step 2 — MBPP+ separately (~5 hr):")
print(f"  python final_run.py --output_dir {FULL_DIR} --model_name Qwen/Qwen3-4B --tasks mbppplus --latent_space_realign")